# 05 — Structured LLM calls (LiteLLM under the hood)

`StructuredLLM` is the LLM adapter; the underlying provider logic is **LiteLLM**. The two presets we ship are thin wrappers over [`litellm.completion`](https://docs.litellm.ai/docs/completion/):

- `FakeProvider` — uses LiteLLM's canonical `mock_response` kwarg ([docs](https://docs.litellm.ai/docs/completion/mock_requests)) to script responses for hermetic tests. No real network call; no bespoke wire format.
- `LMStudioProvider` — routes through LiteLLM's built-in `lm_studio/<model>` provider against a local LM Studio server. **Auto-detects the loaded model** from `/v1/models` if you don't pass one.

`StructuredLLM` itself adds:

1. JSON-schema injection from the Pydantic `response_model`.
2. Validation + **retry-on-validation-error** with the validator message fed back to the model so it can self-correct.
3. Provenance: `llm.call.started` → `llm.call.completed` (with token usage, cost, attempts, budget before/after) → `budget.debited`.
4. Integer-precise micro-USD budget debits (the ledger never goes negative).

**Kernel:** standard `python3`.

In [1]:
import json, tempfile, shutil
from pathlib import Path

from pydantic import BaseModel, Field

from agent_kernel.api import AgentKernel
from agent_kernel.llm import LMStudioProvider, StructuredLLM, LLMCallError
from agent_kernel.models.event import EventType

workspace = Path(tempfile.mkdtemp(prefix='ak-ex05-'))
ak = AgentKernel(workspace)
task = ak.create_task(notebook_path=str(workspace / 'host.ipynb'), kernel_name='python3')
print('task:', task.task_id)

task: task_A3XAMY4MTXEXY974CB5V6CDQ


## 1. Define the response schema

`response_model` is any `pydantic.BaseModel`. Its `model_json_schema()` is what the adapter forwards to the provider.

In [2]:
class Sentiment(BaseModel):
    label: str = Field(description='positive | negative | neutral')
    confidence: float = Field(ge=0.0, le=1.0)

Sentiment.model_json_schema()

{'properties': {'label': {'description': 'positive | negative | neutral',
   'title': 'Label',
   'type': 'string'},
  'confidence': {'maximum': 1.0,
   'minimum': 0.0,
   'title': 'Confidence',
   'type': 'number'}},
 'required': ['label', 'confidence'],
 'title': 'Sentiment',
 'type': 'object'}

## 2. Single happy-path call against `FakeProvider`

`FakeProvider(script=[...])` returns each JSON string from the script in order. We pass `task_id` so the call is accounted to that task's budget.

In [ ]:
import litellm
litellm._turn_on_debug()

In [3]:
provider = LMStudioProvider(
    base_url="http://192.168.86.249:1234/v1",
    cost_usd_micro_per_call=250,   # $0.00025 per call
)
llm = StructuredLLM(provider, agent_kernel=ak)

result = llm.generate(
    messages=[{'role': 'user', 'content': 'Classify: I love this product.'}],
    response_model=Sentiment,
    task_id=task.task_id,
)
print(result)
print('type:', type(result).__name__)

label='Positive' confidence=0.98
type: Sentiment


## 3. Retry-on-validation-error

Script an *invalid* response first, then a valid one. The adapter will see the first response fail Pydantic validation, append the validator's error to the conversation, ask the provider again, and succeed on attempt 2. The `llm.call.completed` event records `attempts=2`.

In [6]:
retry_provider = LMStudioProvider(
    base_url="http://192.168.86.249:1234/v1",
    cost_usd_micro_per_call=100,
)
retry_llm = StructuredLLM(retry_provider, agent_kernel=ak, max_retries=2)

out = retry_llm.generate(
    messages=[{'role': 'user', 'content': 'Classify: this is fine.'}],
    response_model=Sentiment,
    task_id=task.task_id,
)
print(out)

label='Affirmative' confidence=0.98


## 4. Look at the ledger entries

Per call, you should see one `llm.call.started`, one `llm.call.completed`, and one `budget.debited`. The completed event carries usage and the budget_before/after pair, so you can audit cost accounting offline by replaying JSONL alone.

In [7]:
events = ak.list_events(task.task_id)
for e in events:
    if e.event_type in (EventType.llm_call_started, EventType.llm_call_completed, EventType.budget_debited):
        print(f'{e.event_type.value:25s}  {json.dumps(e.payload, sort_keys=True)[:160]}')

llm.call.started           {"call_id": "llm_PCK63K19KB1G93KQ0R6OAIMC", "message_count": 1, "model": null, "provider": "lmstudio", "response_model": "Sentiment"}
llm.call.completed         {"attempts": 1, "call_id": "llm_PCK63K19KB1G93KQ0R6OAIMC", "completion_tokens": 16, "cost_usd_micro": 250, "error": null, "model": null, "prompt_tokens": 16, "p
budget.debited             {"delta": {"bytes_written": 0, "cpu_ms": 0, "llm_input_tokens": 0, "llm_output_tokens": 0, "llm_usd_micro": 250, "spawn_count": 0, "wall_ms": 0}, "reason": "llm
llm.call.started           {"call_id": "llm_22AU6ZJMX77F19F091EPIH0F", "message_count": 1, "model": null, "provider": "lmstudio", "response_model": "Sentiment"}
llm.call.completed         {"attempts": 1, "call_id": "llm_22AU6ZJMX77F19F091EPIH0F", "completion_tokens": 18, "cost_usd_micro": 100, "error": null, "model": null, "prompt_tokens": 15, "p
budget.debited             {"delta": {"bytes_written": 0, "cpu_ms": 0, "llm_input_tokens": 0, "llm_output_tokens":

## 5. Out-of-retries failure

Three invalid responses in a row with `max_retries=2` (i.e. 3 attempts) will raise `LLMCallError`. The cost incurred is still debited; a `llm.call.completed` event is emitted with `status=error`.

In [ ]:
always_bad = FakeProvider(
    script=[
        '{"label": "bad", "confidence": 99}',
        '{"label": "bad", "confidence": 99}',
        '{"label": "bad", "confidence": 99}',
    ],
    cost_usd_micro_per_call=50,
)
bad_llm = StructuredLLM(always_bad, agent_kernel=ak, model='fake-1', max_retries=2)

try:
    bad_llm.generate(
        messages=[{'role': 'user', 'content': '...'}],
        response_model=Sentiment,
        task_id=task.task_id,
    )
except LLMCallError as exc:
    print('expected failure:', str(exc)[:200])

## Pointer to local LM Studio (auto model selection)

Swap the provider line to talk to a real local model. Note that you do **not** have to name the model — `LMStudioProvider` queries `/v1/models` and uses whatever LM Studio currently has loaded, which is what you want for quickstart and ad-hoc testing.

```python
from agent_kernel.llm import LMStudioProvider

provider = LMStudioProvider()                  # default base_url, auto-detect model
if provider.is_reachable():
    print('LM Studio models loaded:', provider.list_models())
    print('will use:               ', provider.resolve_model())
    llm = StructuredLLM(provider, agent_kernel=ak)   # model=None — provider picks
    # … same generate() call …
```

Under the hood `LMStudioProvider` routes the call through LiteLLM's built-in `lm_studio/<model>` provider (it sets `LM_STUDIO_API_BASE` / `LM_STUDIO_API_KEY` and lets LiteLLM handle the transport, retry, and exception normalization).

`is_reachable()` lets you skip cleanly when LM Studio isn't running, which is exactly what the optional `tests/integration/test_m7_llm.py::test_lmstudio_*` test does.

In [ ]:
shutil.rmtree(workspace, ignore_errors=True)